# **Goal**

The purpose of these notebooks is to present some initial experiments to motivate model selection for an interactive app that allows users to experiment with small-medium language model (100M - 1B range).

####**TinyLlama_v1.1**

- <u>Total params</u>: 1.1B. This is the total number of learned scalar values in the entire model: weights, biases, embeddings, output projection, etc.
- <u>Non-embdedding params</u>: 440M. These are the learned parameters in the transformer machinery itself, excluding the token embedding and output vocabulary matrices.
- <u>Number of layers</u>: 22
- <u>Model dimension</u>: 2048. I.e.: Every token is represented internally by a vector of length 1024.
- <u>MLP units per layer:</u> 5632
- <u>Attention Heads</u>: 16 for Q and 8 for KV
- <u>Vocabulary size:</u> 32,000. Amount of tokens in vocabulary.
- <u>Max context window</u>: 32,768 tokens.

Load model

In [ ]:
import transformers
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama_v1.1")
model = AutoModelForCausalLM.from_pretrained("TinyLlama/TinyLlama_v1.1", device_map="auto")

message = "Who are you?"

inputs = tokenizer(
	message,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=50)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True))

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



The first thing I noticed was the lack of a "Welcome to the forum" message. I'm not sure if this is a problem with the forum or the site, but I'm not sure how to fix it.
I


In [ ]:
print(model.config)

LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 1,
  "dtype": "float32",
  "eos_token_id": 2,
  "head_dim": 64,
  "hidden_act": "silu",
  "hidden_size": 2048,
  "initializer_range": 0.02,
  "intermediate_size": 5632,
  "max_position_embeddings": 2048,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 22,
  "num_key_value_heads": 4,
  "pad_token_id": null,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_parameters": {
    "rope_theta": 10000.0,
    "rope_type": "default"
  },
  "tie_word_embeddings": false,
  "transformers_version": "5.16.1",
  "use_cache": true,
  "vocab_size": 32000
}



Prompt experiments. Can play with the following generation parameters:

- `max_new_tokens:` Intenger determining length of response excluding initial prompt.
- `do_sample:` Boolean determining greedy/non-greedy sampling.
- `temperature:` Low temp -> sharp distribution; high temp -> flattened distribution. I.e., how predictable or conservative the model is when sampling tokens.
- `top_k:` Only consider the \(k\) highest-scoring tokens. Default = 50.
- `top_p:` Float 0-1.0. Keep the smallest set of most likely tokens whose probabilities together account for at least \(p\) of the probability mass. Default = 1.
- `repetition_penalty:` How much repeated tokens are discouraged. Default = 1.0 means no penalty.

In [ ]:
message = "How much do you like ice cream?"

inputs = tokenizer(
	message,
	return_tensors="pt",
).to(model.device)


outputs = model.generate(**inputs, max_new_tokens=75)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

[transformers] Both `max_new_tokens` (=75) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



I'm not sure if I'm going to be able to do this. I'm not sure if I'm going to be able to do this. I'm not sure if I'm going to be able to do this. I'm not sure if I'm going to be able to do this. I'm not sure if


In [ ]:
messages = "The capital of France is"

inputs = tokenizer(
	messages,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=True,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.1,
)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True))

[transformers] Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


.
Dating in the 80s - Find a woman in my area! Free to join to find your perfect match. How it works Date a man in the 80s Dating in the 80s.
J


In [ ]:
messages = "The capital of France is"

inputs = tokenizer(
	messages,
	return_tensors="pt",
).to(model.device)


output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=True,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
)

print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True))

[transformers] Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


.
Dating in the 80s - Find a woman in my area! Free to join to find your perfect match. How it works Date a man in the 80s Dating in the 80s.
J


In [ ]:
messages = "The capital of France is"

inputs = tokenizer(
	messages,
	return_tensors="pt",
).to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Input tokens: {input_tokens}")
print(f"Output tokens: {new_tokens}")
print()
print(tokenizer.decode(output[0], skip_special_tokens=True))

[transformers] Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Input tokens: tensor([   1,  450, 7483,  310, 3444,  338])
Output tokens: tensor([29889,    13,  1576, 29871, 29906, 29900, 29896, 29929, 29899, 29906,
        29900, 29906, 29900,  3762,  1629,   338,  1090,  1582,   322,   591,
          526, 24173,   304,  1074,   599,   310,  1749,  8041,  1250,   297,
          278,   770,  8345, 29889,  1334,   526,   884, 24173,   304, 12853,
         1749,   716,  8041,   322, 13175,   304,  1749,  3762, 29889,    13])

The capital of France is.
The 2019-2020 school year is underway and we are excited to see all of our students back in the classroom. We are also excited to welcome our new students and families to our school.



In [ ]:
messages = "The capital of France is"

inputs = tokenizer(
	messages,
	return_tensors="pt",
).to(model.device)

with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits
print(logits.shape)
next_token_logits = logits[0, -1, :]
probs = torch.softmax(next_token_logits, dim=-1)
top_probs, top_ids = torch.topk(probs, k=90)

for prob, token_id in zip(top_probs, top_ids):
    token = tokenizer.decode([token_id.item()])
    print(repr(token), prob.item())

paris_ids = tokenizer.encode(" Paris", add_special_tokens=False)
print(paris_ids)
paris_id = paris_ids[0]
print(probs[paris_id].item())

torch.Size([1, 6, 32000])
'.' 0.1884462833404541
'a' 0.08552291989326477
'\n' 0.05357106775045395
'the' 0.03504415228962898
'and' 0.020812494680285454
'The' 0.020348219200968742
'A' 0.018193813040852547
'' 0.015268363989889622
'at' 0.011066827923059464
'in' 0.009160646237432957
'ag' 0.008525991812348366
'now' 0.008123325183987617
'."' 0.007297700736671686
'taken' 0.006734874099493027
'not' 0.0059581478126347065
'f' 0.005771401803940535
'F' 0.00545575562864542
'caught' 0.0053169988095760345
'are' 0.004862279165536165
'but' 0.004626608919352293
'been' 0.004345192573964596
'right' 0.004193241707980633
'</s>' 0.0038993372581899166
'I' 0.0037768071051687002
'T' 0.0036905629094690084
'being' 0.0033584123011678457
'a' 0.0033229421824216843
's' 0.0031156239565461874
'to' 0.003078240668401122
'D' 0.003050059312954545
'De' 0.0030348452273756266
'flag' 0.0029970004688948393
'there' 0.0028706344310194254
'And' 0.002769799903035164
'by' 0.002693584654480219
'called' 0.0024785883724689484
'read' 0.0

Here we are just printing the internal shape of one layer of the model.


In [ ]:
print(model.model.layers[0].mlp)

LlamaMLP(
  (gate_proj): Linear(in_features=2048, out_features=5632, bias=False)
  (up_proj): Linear(in_features=2048, out_features=5632, bias=False)
  (down_proj): Linear(in_features=5632, out_features=2048, bias=False)
  (act_fn): SiLUActivation()
)


In [ ]:
model.config.intermediate_size

5632

The input vector representation is 2048-dimensional. The intermediate dimension is 3072. Note that the Qwen models use more sophisticated gated MLPS.

There are 2 separate sets of 3072 MLP units that receive the input. Each of these produces a a transformed and expanded 3072-d vector called the gate projection and the up projection. The gate projection goes through activation using SiLU (Sigmoid Linear Unit). Then this activated projection is multiplied element-wise with the up-projection, and these new values are transformed back into a 1024-d representation by another learned matrix. This representation will continue through the transformer block and subsequent layers.

Note: $\text{SiLU}(x) = x ⋅ σ(x)$, where $σ(x)$ is the sigmoid function $\frac{1}{1+e^{-x}}$.

**MLP Unit Ablation Experiments**

In [ ]:
import torch

def add_mlp_ablation(model, layer_idx, proportion, seed=42):

    # MLP in layer layer_idx
    mlp = model.model.layers[layer_idx].mlp

    num_units = model.config.intermediate_size
    num_ablate = int(proportion * num_units)

    # Seed RNG
    generator = torch.Generator()
    generator.manual_seed(seed)

    # Pick 'num_ablate' indexes for MLP units at random
    indices = torch.randperm(
        num_units,
        generator=generator
    )[:num_ablate]

    def ablation_hook(module, inputs):
        x = inputs[0].clone()
        indices_device = indices.to(x.device)
        x[..., indices_device] = 0
        return (x,) + inputs[1:]

    handle = mlp.down_proj.register_forward_pre_hook(ablation_hook)

    return handle, indices, layer_idx, proportion

Here we try zeroing the outputs of different proportions of the first (0th) layer. Notice gradual degradation in output coherence.

Baseline

In [ ]:
handle, indices, layer_id, proportion = add_mlp_ablation(
    model,
    layer_idx=0,
    proportion=0.0,
)

messages = "The capital of France is"

inputs = tokenizer(
	messages,
	return_tensors="pt",
).to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Input tokens: {input_tokens}")
print(f"Output tokens: {new_tokens}")
print()
print(f"Zeroed activations in {proportion}% of units in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

[transformers] Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Input tokens: tensor([   1,  450, 7483,  310, 3444,  338])
Output tokens: tensor([29889,    13,  1576, 29871, 29906, 29900, 29896, 29929, 29899, 29906,
        29900, 29906, 29900,  3762,  1629,   338,  1090,  1582,   322,   591,
          526, 24173,   304,  1074,   599,   278,   716, 17240,   297,  1749,
          770, 18901, 29991,  1334,   505,   263,  2107,  2318,   310,  8041,
        29892, 27335, 29892,   322, 13925,  5144,  1058,   674,   367,  1985])

Zeroed activations in 0.0% of units in layer 0
The capital of France is.
The 2019-2020 school year is underway and we are excited to see all the new faces in our classrooms! We have a great group of students, teachers, and staff members who will be working


$25\%$ of first layer

In [ ]:
handle, indices, layer_id, proportion = add_mlp_ablation(
    model,
    layer_idx=0,
    proportion=0.25,
)

messages = "The capital of France is"

inputs = tokenizer(
	messages,
	return_tensors="pt",
).to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(f"Zeroed activations in {proportion*100}% of units in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

[transformers] Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Inputs:tensor([   1,  450, 7483,  310, 3444,  338])
Outputs: tensor([29889,    13,  1576, 29871, 29906, 29900, 29896, 29947, 21581,  2787,
         6536,   471,   278, 29871, 29906, 29896,   303, 21581,  2787,  6536,
        29892,   385,  6121,  5733, 14743,   640,  2868,   491,   278,  1757,
        29915, 29879,  4797, 10907,   310,   278,  4509, 27733,   310, 21581,
         2748,  1432,  3023,  2440, 29889,   739,  3614,  2058,   297, 12710])

Zeroed activations in 25.0% of units in layer 0
The capital of France is.
The 2018 FIFA World Cup was the 21st FIFA World Cup, an international football tournament contested by the men's national teams of the member associations of FIFA once every four years. It took place in Russia


$50\%$ of first layer

In [ ]:
handle, indices, layer_id, proportion = add_mlp_ablation(
    model,
    layer_idx=0,
    proportion=0.5,
)

messages = "The capital of France is"

inputs = tokenizer(
	messages,
	return_tensors="pt",
).to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(f"Zeroed activations in {proportion*100}% of units in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

[transformers] Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Inputs:tensor([   1,  450, 7483,  310, 3444,  338])
Outputs: tensor([29889,    13,  1576, 29871, 29906, 29900, 29896, 29929, 29899, 29906,
        29900, 29906, 29900,  3762,  1629,   338,   263,   716,   697,   363,
          278,  8041,   472,   278,  3762, 29892,   322,   896,   526, 24173,
          304,   367,  1250,   297,   770, 18901, 29889,  2688,   505,  1063,
         1985,  2898,   373,  1009,  3566,  1860,   322,  9279,   975,   278])

Zeroed activations in 50.0% of units in layer 0
The capital of France is.
The 2019-2020 school year is a new one for the students at the school, and they are excited to be back in classrooms. They have been working hard on their assignments and projects over the


$75\%$ of first layer

In [ ]:
handle, indices, layer_id, proportion = add_mlp_ablation(
    model,
    layer_idx=0,
    proportion=0.75,
)

messages = "The capital of France is"

inputs = tokenizer(
	messages,
	return_tensors="pt",
).to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.8,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(f"Zeroed activations in {proportion*100}% of units in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

[transformers] Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Inputs:tensor([   1,  450, 7483,  310, 3444,  338])
Outputs: tensor([29889,    13,  1576, 29871, 29906, 29900, 29896, 29929, 29899, 29906,
        29900, 29906, 29900,  3762,  1629,   674,   367,   278,   937,   931,
          393,   278,  3762,  6474,   756,  1063,  2221,   304,  5957,   263,
         2989,  3464,   322, 12875,   297,   967, 16256, 12906,   398, 29892,
         3704,   263,   716, 10466,  1824,   363,   867,  3076,  4832,  1549])

Zeroed activations in 75.0% of units in layer 0
The capital of France is.
The 2019-2020 school year will be the first time that the school district has been able to offer a full range and variety in its curriculum, including a new science program for grades six through


$100\%$ of first layer

In [ ]:
handle, indices, layer_id, proportion = add_mlp_ablation(
    model,
    layer_idx=0,
    proportion=1.0,
)

messages = "The capital of France is"

inputs = tokenizer(
	messages,
	return_tensors="pt",
).to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.8,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(f"Zeroed activations in {proportion*100}% of units in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

[transformers] Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Inputs:tensor([   1,  450, 7483,  310, 3444,  338])
Outputs: tensor([  263, 29889,    13,   319,  1213, 29871, 29906, 29908,   376,  1642,
          891,   396,  1763,   732,  7495,   323,  3446, 19988,  6093,   350,
        17067,   317, 13315,   304,  6850,  1806, 29903,  7488, 29411,  7997,
         5974,  1920,   910, 13611,  7803, 15472,  1048, 20147,  1619,  8168,
        20377,   739,   931,   306, 29945,   601,  1023,  2246,  3942,   413])

Zeroed activations in 100.0% of units in layer 0
The capital of France is a.
 A." 2" "". | # To @ TO T THTT THE B TE S IT to STITS Top Toy Big Time Te This About Two Tree about Twitter My Tw Tokyo It time I5io two top family k


**Attention Head Ablation Experiments**

First let's check the attention heads

In [ ]:
print(model.model.layers[0])

LlamaDecoderLayer(
  (self_attn): LlamaAttention(
    (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
    (k_proj): Linear(in_features=2048, out_features=256, bias=False)
    (v_proj): Linear(in_features=2048, out_features=256, bias=False)
    (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
  )
  (mlp): LlamaMLP(
    (gate_proj): Linear(in_features=2048, out_features=5632, bias=False)
    (up_proj): Linear(in_features=2048, out_features=5632, bias=False)
    (down_proj): Linear(in_features=5632, out_features=2048, bias=False)
    (act_fn): SiLUActivation()
  )
  (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
  (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
)


We see that the attention mechanism expects a 2048d input (the token embedding or output from the previous block). It then computes a 2048-d vector q_proj, and two 256-d vectors, k_proj and v_proj. This corresponds to calculating its query, key, and value vectors. Note that there are are 16 Q-heads and 8-KV heads, so each key/value head is shared by two query heads. After computing attention scores using these 3 vectors, the attention mechanism produces 32 query-head outputs, each with dimension 64. This 2048 output goes into the o-projection, a learned projection from the concatenated attention head contributions, to the final attention contribution with dimension 2048.

In [ ]:
print(model.model.layers[0].self_attn.head_dim)

64


In [ ]:
print(model.model.layers[0].self_attn.o_proj)

Linear(in_features=2048, out_features=2048, bias=False)


In [ ]:
print(model.model.layers[0].self_attn.o_proj.in_features)

2048


In [ ]:
model.config.num_hidden_layers

22

In [ ]:
model.config.hidden_size

2048

In [ ]:
model.config.num_attention_heads

32

Attention head ablation function

In [ ]:
def add_head_ablation(model, head_indices, layer_idx):

    #Number of hidden layers
    num_layers = model.config.num_hidden_layers
    #Number of Query attention heads
    num_heads = model.config.num_attention_heads

    if not 0 <= layer_idx < num_layers:
        raise ValueError(f"layer_idx must be between 0 and {num_layers - 1}")
    for head_idx in head_indices:
        if not 0 <= head_idx < num_heads:
            raise ValueError(f"head_idx must be between 0 and {num_heads - 1}")

    #Projection dimension per attention head
    head_dim = model.model.layers[layer_idx].self_attn.head_dim

    # Load final attention module
    attention_dense = model.model.layers[layer_idx].self_attn.o_proj

    def head_ablation_hook(module, inputs):
        x = inputs[0].clone()

        for head_idx in head_indices:
            #compute indices corresponding to specific head
            start = head_idx * head_dim
            end = start + head_dim

            #Select indices in the head outputs vector corresponding to chosen head and zero them
            x[..., start:end] = 0

        return (x,) + inputs[1:]

    #Note this is a pre-hook because we want to intervene components before the o-projection forward pass
    handle = attention_dense.register_forward_pre_hook(head_ablation_hook)

    return handle, head_indices, layer_idx

Here we try zeroing the outputs of different numbers of heads of the first (0th) layer.

Remove 1 head from layer 0

In [ ]:
handle, head_ids, layer_id = add_head_ablation(
    model,
    head_indices=[0],
    layer_idx=0,
)

messages = "The capital of France is"

inputs = tokenizer(
	messages,
	return_tensors="pt",
).to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Input tokens: {input_tokens}")
print(f"Output tokens: {new_tokens}")
print()
print(f"Zeroed head {[x for x in head_ids]} in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

[transformers] Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Input tokens: tensor([   1,  450, 7483,  310, 3444,  338])
Output tokens: tensor([  263,   304, 29889,    13,  1576, 29871, 29906, 29900, 29896, 29947,
        21581,  2787,  6536,   471,   278, 29871, 29906, 29896,   303, 21581,
         2787,  6536, 29892,   385,  6121,  5733, 14743,   640,  2868,   491,
          278,  1757, 29915, 29879,  4797, 10907,   310,   278,  4509, 27733,
          310, 21581,  2748,  1432,  3023,  2440, 29889,   739,  3614,  2058])

Zeroed head [0] in layer 0
The capital of France is a to.
The 2018 FIFA World Cup was the 21st FIFA World Cup, an international football tournament contested by the men's national teams of the member associations of FIFA once every four years. It took place


Ablate 4 attention heads in layer 0

---



In [ ]:
to_ablate = int(model.config.num_attention_heads*0.25)

handle, head_ids, layer_id = add_head_ablation(
    model,
    head_indices=[x for x in range(to_ablate)],
    layer_idx=0,
)

messages = "The capital of France is"

inputs = tokenizer(
	messages,
	return_tensors="pt",
).to(model.device)


output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(f"Zeroed head {[x for x in head_ids]} in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

[transformers] Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Inputs:tensor([   1,  450, 7483,  310, 3444,  338])
Outputs: tensor([  263, 29889,    13,  1576, 29871, 29896, 29929, 29947, 29900, 29879,
          471,   278,   316,  6332,   310,   278, 29871, 29941,  5499,  2913,
         5046, 29892,   322,   278, 29871, 29906,   299,  2913,  8175, 29889,
          450,   316,  6332,  4689,   373, 24211, 29892,  5490, 29871, 29896,
          303, 29892, 29871, 29896, 29929, 29947, 29900, 29892,  9698,   373])

Zeroed head [0, 1, 2, 3, 4, 5, 6, 7] in layer 0
The capital of France is a.
The 1980s was the decade of the 3rd space age, and the 2nd space race. The decade began on Saturday, January 1st, 1980, ended on


Ablate 8 attention heads in layer 0

In [ ]:
to_ablate = int(model.config.num_attention_heads*0.5)

handle, head_ids, layer_id = add_head_ablation(
    model,
    head_indices=[x for x in range(to_ablate)],
    layer_idx=0,
)

messages = "The capital of France is"

inputs = tokenizer(
	messages,
	return_tensors="pt",
).to(model.device)


output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.8,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(f"Zeroed head {[x for x in head_ids]} in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

[transformers] Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Inputs:tensor([   1,  450, 7483,  310, 3444,  338])
Outputs: tensor([   13,  1576,    13, 29933,   768,   305,   314, 29871, 29906, 29900,
        29896, 29947, 29929, 29955, 29953, 29941, 29946, 29945, 29892, 29871,
        30211,    13, 29909, 29871,   229,   131, 30098,     2])

Zeroed head [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15] in layer 0
The capital of France is
The
Britcham 2018976345, ©
A ��…


Ablate 12 attention heads in layer 0

In [ ]:
to_ablate = int(model.config.num_attention_heads*0.75)

handle, head_ids, layer_id = add_head_ablation(
    model,
    head_indices=[x for x in range(to_ablate)],
    layer_idx=0,
)

messages = "The capital of France is"

inputs = tokenizer(
	messages,
	return_tensors="pt",
).to(model.device)


output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.8,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Input tokens: {input_tokens}")
print(f"Output tokens: {new_tokens}")
print()
print(f"Zeroed head {[x for x in head_ids]} in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

[transformers] Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Input tokens: tensor([   1,  450, 7483,  310, 3444,  338])
Output tokens: tensor([15048,   411,   351,   314,   287,   615,  2447,   265, 26103,  5513,
        15945, 29908,  1159,   387,    13,  4706,   313, 29896, 29897,   462,
        29871,   518, 29900, 29962,  1678, 29906, 29889, 29941, 29946, 29945,
        29953, 29955, 29947, 29929,   259,   869,  2732,   268,    13, 29898,
        29896, 29892, 29906,  7671,  6173, 18660, 11891, 18660,   467,  2492])

Zeroed head [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23] in layer 0
The capital of France isotted withagamedftederonege)""""")eg
        (1)                  [0]   2.3456789   .png    
(1,2,.jpggif(.gif).img


Ablate 16 (all) attention heads in layer 0

In [ ]:
to_ablate = int(model.config.num_attention_heads)

handle, head_ids, layer_id = add_head_ablation(
    model,
    head_indices=[x for x in range(to_ablate)],
    layer_idx=0,
)

messages = "The capital of France is"

inputs = tokenizer(
	messages,
	return_tensors="pt",
).to(model.device)


output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.5,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Input tokens: {input_tokens}")
print(f"Output tokens: {new_tokens}")
print()
print(f"Zeroed head {[x for x in head_ids]} in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

[transformers] Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Input tokens: tensor([   1,  450, 7483,  310, 3444,  338])
Output tokens: tensor([ 3428,   810,  6308, 18291, 28818,  3363, 22382, 29997, 10557, 27956,
        17191, 19107, 20537, 30000, 10522,  7781, 15981,  2470, 29948,  8428,
         3621,  6948,   609,  1657, 16846, 20634,  1365, 11902,  1960,  2834,
         6454, 22396,  2745, 26207,  4909,  6715,  7443, 13298,  4673, 14124,
        29887, 16926,   313, 10356, 16035, 25132, 26411, 26974,  7365, 30069])

Zeroed head [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31] in layer 0
The capital of France is tillags Afroustgatäncommonsö Africa Tillvil honour Nigerè Tagaga Niourséunter Londunsontuntdamossavo Wolves life tomorrow untilöffentlichville Odongo Domin Open Łgugs (UNT explo Dresden Valent Gün Krô


Ablate all attention heads in layer 11

In [ ]:
to_ablate = int(model.config.num_attention_heads)

handle, head_ids, layer_id = add_head_ablation(
    model,
    head_indices=[x for x in range(to_ablate)],
    layer_idx=11,
)

messages = "The capital of France is"

inputs = tokenizer(
	messages,
	return_tensors="pt",
).to(model.device)


output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(f"Zeroed head {[x for x in head_ids]} in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

[transformers] Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Inputs:tensor([   1,  450, 7483,  310, 3444,  338])
Outputs: tensor([  278, 29889,    13,  1576, 29871, 29906, 29900, 29896, 29947, 21581,
         2787,  6536,   471,   278, 29871, 29906, 29896,   303, 21581,  2787,
         6536, 29892,   385,  6121,  5733, 14743,   640,  2868,   491,   278,
         1757, 29915, 29879,  4797, 10907,   310,   278,  4509, 27733,   310,
        21581,  2748,  1432,  3023,  2440, 29889,   739,  3614,  2058,   297])

Zeroed head [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31] in layer 11
The capital of France is the.
The 2018 FIFA World Cup was the 21st FIFA World Cup, an international football tournament contested by the men's national teams of the member associations of FIFA once every four years. It took place in


Ablate heads in every layer

Baseline

In [ ]:
messages = "The capital of France is"

inputs = tokenizer(
	messages,
	return_tensors="pt",
).to(model.device)


num_layers = model.config.num_hidden_layers
ablations = { x: [] for x in range(num_layers) }

handles = []

try:
    for layer_idx, heads in ablations.items():
        handle, _, _ = add_head_ablation(
            model,
            head_indices=heads,
            layer_idx=layer_idx
        )
        handles.append(handle)

    output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.2,
    )

finally:
    for handle in handles:
        handle.remove()

print(f"Baseline")
print(tokenizer.decode(output[0], skip_special_tokens=True))

[transformers] Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Baseline
The capital of France is.
The 2019-2020 school year is underway and we are excited to see all the new faces in our classrooms! We have a great group of students, teachers, and staff members who will be working


In [ ]:
messages = "The capital of France is"

inputs = tokenizer(
	messages,
	return_tensors="pt",
).to(model.device)


num_layers = model.config.num_hidden_layers
ablations = { x: [0] for x in range(num_layers) }

handles = []

try:
    for layer_idx, heads in ablations.items():
        handle, _, _ = add_head_ablation(
            model,
            head_indices=heads,
            layer_idx=layer_idx
        )
        handles.append(handle)

    output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.2,
    )

finally:
    for handle in handles:
        handle.remove()

print(f"Zeroed head 0 in all layers")
print(tokenizer.decode(output[0], skip_special_tokens=True))

[transformers] Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Zeroed head 0 in all layers
The capital of France is
The 2018-2019 school year is underway and the students are excited to learn. This year, we have a new principal at our school, Mr.
The 2018-201


In [ ]:
to_ablate = int(model.config.num_attention_heads*0.25)

messages = "The capital of France is"

inputs = tokenizer(
	messages,
	return_tensors="pt",
).to(model.device)


num_layers = model.config.num_hidden_layers
ablations = { x: [head for head in range(to_ablate)] for x in range(num_layers) }

handles = []

try:
    for layer_idx, heads in ablations.items():
        handle, _, _ = add_head_ablation(
            model,
            head_indices=heads,
            layer_idx=layer_idx
        )
        handles.append(handle)

    output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.8,
    )

finally:
    for handle in handles:
        handle.remove()

print(f"Zeroed heads 0,1,2,3 in all layers")
print(tokenizer.decode(output[0], skip_special_tokens=True))

[transformers] Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Zeroed heads 0,1,2,3 in all layers
The capital of France is the capital for the capital.
The French are a proud people, and the more I have to say about them in this book will be found on pains-departage.fr
This is not an exhaustive list; it's


In [ ]:
to_ablate = int(model.config.num_attention_heads*0.5)

messages = "The capital of France is"

inputs = tokenizer(
	messages,
	return_tensors="pt",
).to(model.device)


num_layers = model.config.num_hidden_layers
ablations = { x: [head for head in range(to_ablate)] for x in range(num_layers) }

handles = []

try:
    for layer_idx, heads in ablations.items():
        handle, _, _ = add_head_ablation(
            model,
            head_indices=heads,
            layer_idx=layer_idx
        )
        handles.append(handle)

    output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.8,
    )

finally:
    for handle in handles:
        handle.remove()

print(f"Zeroed heads 0,1,2,3,4,5,6,7 in all layers")
print(tokenizer.decode(output[0], skip_special_tokens=True))

[transformers] Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Zeroed heads 0,1,2,3,4,5,6,7 in all layers
The capital of France is anda.
Andrewrotectofficiallysaysomething, which was the first published in 1980000020000000000000000000


In [ ]:
to_ablate = int(model.config.num_attention_heads)

messages = "The capital of France is"

inputs = tokenizer(
	messages,
	return_tensors="pt",
).to(model.device)


num_layers = model.config.num_hidden_layers
ablations = { x: [head for head in range(to_ablate)] for x in range(num_layers) }

handles = []

try:
    for layer_idx, heads in ablations.items():
        handle, _, _ = add_head_ablation(
            model,
            head_indices=heads,
            layer_idx=layer_idx
        )
        handles.append(handle)

    output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.8,
    )

    input_length = inputs["input_ids"].shape[1]
    new_tokens = output[0][input_length:]
    input_tokens = output[0][:input_length]

finally:
    for handle in handles:
        handle.remove()

print(f"Output tokens: {new_tokens}")
print()
print(f"Zeroed all heads in all layers")
print(tokenizer.decode(output[0], skip_special_tokens=True))

[transformers] Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Output tokens: tensor([ 9523,  3322,  4618, 29878,  7261,   373,  8777, 29915, 31799, 29899,
        13559,   663, 29889,    13, 20448,  2478, 29892, 20729,  1107,   433,
          381,   328,  7532, 29889,    13, 13922,   341,  4992,   538, 29892,
        17943, 29871, 28010,   411, 22792,   787, 29882, 17700, 13211,  5834,
          322,  1786,  8197,  5460,  1630,  8639, 29889,    13, 28361,   366])

Zeroed all heads in all layers
The capital of France is staunchangedrounded onrowser'Ě- становиence.
 Kontrolacies, Kirchabellairadrift.
Memento Moneyard, Category vés with summonsh Coast Guardians and воинтермораль.
 Zobacz you


In [ ]:
to_ablate = int(model.config.num_attention_heads)

messages = "The capital of France is"

inputs = tokenizer(
	messages,
	return_tensors="pt",
).to(model.device)


num_layers = model.config.num_hidden_layers
ablations = { x: [head for head in range(to_ablate)] for x in range(num_layers) }

handles = []

try:
    for layer_idx, heads in ablations.items():
        handle, _, _ = add_head_ablation(
            model,
            head_indices=heads,
            layer_idx=layer_idx
        )
        handles.append(handle)

    output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.2,
    )

    input_length = inputs["input_ids"].shape[1]
    new_tokens = output[0][input_length:]
    input_tokens = output[0][:input_length]

finally:
    for handle in handles:
        handle.remove()

print(f"Output tokens: {new_tokens}")
print()
print(f"Zeroed all heads in all layers")
print(tokenizer.decode(output[0], skip_special_tokens=True))

[transformers] Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Output tokens: tensor([ 9523,  3322,  4618, 29878,  7261,   373,  8777, 29915, 31799, 29899,
        13559,   663, 29889,    13, 20448, 29899, 28010,   310, 20729,  1107,
          433,   381,   328,  7532, 29889,    13, 13922,   341,  4992,   538,
        29892, 17943, 29871, 17700, 13211,  5834, 29889,    13, 28361,   366,
         1786,  8197,  5460,  1630,  8639, 29889,    13, 17835,   514,   487])

Zeroed all heads in all layers
The capital of France is staunchangedrounded onrowser'Ě- становиence.
 Kontrola-vés of Kirchabellairadrift.
Memento Moneyard, Category  Coast Guardians.
 Zobacz you воинтермораль.
 Становellore
